In [ ]:
# Install Telethon
# !pip install telethon

# !pip install python-dotenv
# !pip install tqdm

In [ ]:
import os
import json
import tqdm
import asyncio
import requests
from telethon import TelegramClient
# from datetime import datetime, UTC #3.11
from datetime import datetime, timezone

In [ ]:
# %load_ext dotenv
# %dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=False)

In [ ]:
# Replace with your own values
api_id = os.getenv("API_ID")
api_hash = os.getenv("API_HASH")

# Session name (creates a local .session file)
session_name = 'telegram_session'

client = TelegramClient(session_name, api_id, api_hash)

### Service Routines

In [ ]:
# news_channel_list = [
# 1052247604, # LIGA.net
# 1409807527, # Труха⚡️Жесть 18+
# 1199360700, # Труха⚡️Україна
# 1231519967, # НЕВЗОРОВ
# 1245586711, # Доброго вечора, ми з Дніпра👋🏻
# 1487406725, # Труха⚡️Дніпро | Новини
# 1933547006, # Israel⚡️Труха
# 1106144694, # ФЕЙГИН LIVE
# 1842583506, # Наші у світі 🌍🇺🇦
# 1940511315, # Наші в Канаді 🇨🇦🇺🇦
# 1075023847, # Дежурный по Израилю
# 1262084660, # Gulagu.net ГУЛАГу-НЕТ! 🕊
# 1982968478, # Наші в Америці 🇺🇸🇺🇦
# 2031233106, # David Gendelman
# ]

In [ ]:
# Print the list of channels and their IDs
async def print_channel_list():
    dialogs = await client.get_dialogs()
    # print(dialogs)

    for dialog in dialogs:
        print(f"{dialog.entity.id}, # {dialog.name}")

In [ ]:
# Save the list of available channels and their IDs to a JSON file
async def save_channel_list_to_json(file_name="telegram_channels.json"):
    dialogs = await client.get_dialogs()
    channel_list = [{"id": dialog.entity.id, "name": dialog.name} for dialog in dialogs]

    with open(file_name, "w", encoding="utf-8") as f:
        json.dump(channel_list, f, ensure_ascii=False, indent=4)

    print(f"Channel list saved to {file_name}")

In [ ]:
# Read channel list from a JSON file
def load_channel_list_from_json(file_name="telegram_channels.json"):
    with open(file_name, 'r') as f:
        channels = json.load(f)

    return [channel['id'] for channel in channels]

In [ ]:
async def write_unread_messages(news_channel_list):
    # datetime.now(UTC).strftime("%Y-%m-%d_%H-%M-%S") #3.11
    export_timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S")
    file_name = f"News_Extract_{export_timestamp}.txt"
    print(f"Exporting unread messages to: {file_name}")

    dialogs = await client.get_dialogs()
    #print(dialogs)

    with open(file_name, "w", encoding="utf-8") as f:

        for dialog in dialogs:

            if dialog.entity.id not in news_channel_list:
                continue

            print(f"Channel: {dialog.name} | Unread messages: {dialog.unread_count} | Identity: {dialog.entity.id}")

            if dialog.unread_count > 0:
                messages = await client.get_messages(
                    dialog.entity,
                    limit=dialog.unread_count
                )

                # Reverse so oldest unread appears first
                for msg in reversed(messages):
                    if msg.message:
                        f.write(f"Chat: {dialog.name} | Date: {msg.date.strftime('%Y-%m-%d %H:%M:%S')}\n")
                        f.write(f"{msg.message}\n")
                        f.write(f"{'-' * 60}\n\n")
                        # print(f"Date: {msg.date}")
                        # print(f"Text: {msg.message}")
                        # print()

                    await msg.mark_read()

### Execute

In [ ]:
await client.start()
print("Logged into Telegram.\n")

In [ ]:
news_channel_list = load_channel_list_from_json()
await write_unread_messages(news_channel_list)

In [ ]:
# datetime.now(UTC).strftime("%Y-%m-%d_%H-%M-%S") #3.11
# export_timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S")
# file_name = f"telegram_channels_{export_timestamp}.json"

# await save_channel_list_to_json(file_name)

In [ ]:
await client.disconnect()

### Legacy Code

In [ ]:
### End of processing
# assert 0, 'stop'

In [ ]:
# async def read_unread_messages():
#     await client.start()

#     print("Logged into Telegram.\n")

#     dialogs = await client.get_dialogs()

#     found_unread = False

#     for dialog in dialogs:
#         unread_count = dialog.unread_count

#         if unread_count > 0:
#             found_unread = True

#             print("=" * 60)
#             print(f"Chat: {dialog.name}")
#             print(f"Unread messages: {unread_count}")
#             print("-" * 60)

#             messages = await client.get_messages(
#                 dialog.entity,
#                 limit=unread_count
#             )

#             # Reverse so oldest unread appears first
#             for msg in reversed(messages):

#                 sender = await msg.get_sender()

#                 if sender:
#                     sender_name = getattr(sender, 'first_name', None)

#                     if not sender_name:
#                         sender_name = getattr(sender, 'title', 'Unknown')
#                 else:
#                     sender_name = "Unknown"

#                 text = msg.message or "[Non-text message]"

#                 print(f"{sender_name}: {text}")

#             print()

#     if not found_unread:
#         print("No unread messages found.")

#     await client.disconnect()

In [ ]:
# Run inside Jupyter notebook
# await read_unread_messages()